In [1]:
from unsloth import FastLanguageModel
from vllm import SamplingParams

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 08-27 11:11:14 [__init__.py:241] Automatically detected platform cuda.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
max_seq_length = 2048
lora_rank = 32

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Base",
    max_seq_length = max_seq_length,
    load_in_4bit = False,
    fast_inference = True,
    adapter_id = "ZhengjunHUO/Qwen3-4B-Reasoning-LoRA",
    gpu_memory_utilization = 0.7,
)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Unsloth: Patching vLLM v1 graph capture
Unsloth: Patching vLLM v0 graph capture
==((====))==  Unsloth 2025.8.9: Fast Qwen3 patching. Transformers: 4.55.2. vLLM: 0.10.1.
   \\   /|    NVIDIA GeForce RTX 4070 Ti SUPER. Num GPUs = 1. Max memory: 15.577 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.31. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/Qwen3-4B-Base with actual GPU utilization = 66.36%
Unsloth: Your GPU has CUDA compute capability 8.9 with VRAM = 15.58 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 2048. Num Sequences = 160.
Unsloth: vLLM's KV Cache can use up to 3.27 GB. Also swap space = 6 GB.
Unsloth: Not an error, but `device` is not supported in vLLM. Skipping.
INFO 08-27 11:11:19 [utils.py:326] non-defaul

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 08-27 11:11:26 [default_loader.py:262] Loading weights took 0.85 seconds
INFO 08-27 11:11:26 [punica_selector.py:19] Using PunicaWrapperGPU.
INFO 08-27 11:11:27 [gpu_model_runner.py:2007] Model loading took 7.7565 GiB and 1.400285 seconds
INFO 08-27 11:11:34 [backends.py:548] Using cache directory: /home/huo/.cache/vllm/torch_compile_cache/ae7bd6dc49/rank_0_0/backbone for vLLM's torch.compile
INFO 08-27 11:11:34 [backends.py:559] Dynamo bytecode transform time: 7.34 s
INFO 08-27 11:11:41 [backends.py:161] Directly load the compiled graph(s) for dynamic shape from the cache, took 5.371 s
INFO 08-27 11:11:42 [monitor.py:34] torch.compile takes 7.34 s in total
INFO 08-27 11:11:43 [gpu_worker.py:276] Available KV cache memory: 1.66 GiB
INFO 08-27 11:11:43 [kv_cache_utils.py:849] GPU KV cache size: 12,112 tokens
INFO 08-27 11:11:43 [kv_cache_utils.py:853] Maximum concurrency for 2,048 tokens per request: 5.91x
INFO 08-27 11:11:43 [vllm_utils.py:643] Unsloth: Running patched vLLM v1 `ca

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 43/43 [00:05<00:00,  7.25it/s]

INFO 08-27 11:11:49 [gpu_model_runner.py:2708] Graph capturing finished in 6 secs, took 0.51 GiB
INFO 08-27 11:11:49 [vllm_utils.py:650] Unsloth: Patched vLLM v1 graph capture finished in 6 secs.


INFO 08-27 11:11:50 [core.py:214] init engine (profile, create kv cache, warmup model) took 22.73 seconds
INFO 08-27 11:11:50 [llm.py:298] Supported_tasks: ('generate',)
Unsloth: Just some info: will skip parsing ['pre_feedforward_layernorm', 'post_feedforward_layernorm']
Unsloth: Just some info: will skip parsing ['pre_feedforward_layernorm', 'post_feedforward_layernorm']


In [3]:
FastLanguageModel.for_inference(model) # Enables optimized inference

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560, padding_idx=151654)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen3

In [4]:
reasoning_start = "<start_breaking_down>"
reasoning_end   = "<end_breaking_down>"
solution_start  = "<SOLUTION>"
solution_end    = "</SOLUTION>"

system_prompt = \
f"""You are given a problem.
Think about the problem and provide your working out.
Place it between {reasoning_start} and {reasoning_end}.
Then, provide your solution between {solution_start}{solution_end}"""

chat_template = \
    "{% if messages[0]['role'] == 'system' %}"\
        "{{ messages[0]['content'] + eos_token }}"\
        "{% set loop_messages = messages[1:] %}"\
    "{% else %}"\
        "{{ '{system_prompt}' + eos_token }}"\
        "{% set loop_messages = messages %}"\
    "{% endif %}"\
    "{% for message in loop_messages %}"\
        "{% if message['role'] == 'user' %}"\
            "{{ message['content'] }}"\
        "{% elif message['role'] == 'assistant' %}"\
            "{{ message['content'] + eos_token }}"\
        "{% endif %}"\
    "{% endfor %}"\
    "{% if add_generation_prompt %}{{ '{reasoning_start}' }}"\
    "{% endif %}"

chat_template = chat_template\
    .replace("'{system_prompt}'",   f"'{system_prompt}'")\
    .replace("'{reasoning_start}'", f"'{reasoning_start}'")
tokenizer.chat_template = chat_template

In [7]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "What is the sqrt of 101?"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = False,
)

sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 2048,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [8]:
output

'Alright, so I need to find the square root of 101. But before I jump into calculating, I should understand what a square root means. The square root of a number x is another number r such that r^2 = x. So essentially, I\'m looking for a number which, when multiplied by itself, gives me 101.\n\nFirst, I should think about whether 101 is a perfect square. A perfect square is an integer that is the square of another integer. Let\'s see: \n\n- 10^2 = 100\n- 11^2 = 121\n\nSince 101 is between 100 and 121, it\'s not a perfect square, which means its square root will be an irrational number. Irrational numbers are real numbers that cannot be expressed as a fraction of two integers.\n\nGiven that 101 isn\'t a perfect square, I can\'t find an exact square root using simple methods like prime factorization. For non-perfect squares like 101, we typically use approximation methods to find the square root.\n\nOne common approximation method is to use the average method or more advanced techniques 